# Burfisher 3e — ME9A: línea de base 2010-2050 con PIB real exógeno

Proyecta la economía a 2050 antes de cualquier shock de política. El crecimiento del
PIB real (USA +109,6%, ROW +284,5%), de la población y de los factores son dato; el
modelo calcula la **productividad** que hace falta para llegar a ese PIB.

En GEMPACK (`ME9A.EXP`, `climatechange.prm`) el cierre es `swap qgdp(reg) = aoreg(reg)`:
el PIB pasa a exógeno y el shifter regional de productividad `aoreg` (el mismo en las 3
actividades) a endógeno. Aquí se arma modificando dos bloques **sólo en este notebook**
con `@overwrite`.

In [ ]:
from pathlib import Path

from pyomo.environ import value

from equilibria._local_refs import nus333_dir
from equilibria.blocks.gtap import ClosureBlock, ShockBlock, overwrite
from equilibria.templates.gtap import GTAPParameters
from equilibria.templates.gtap.gtap_block_model import build_block_model
from equilibria.templates.gtap.gtap_contract import GTAPClosureConfig
from equilibria.templates.gtap.gtap_multiperiod_driver import solve_multiperiod
from equilibria.templates.gtap.instruments import fix_instrument_shock

## 1. Datos: la base nus333 (2 regiones × 3 sectores) y el `.prm` del ejercicio

In [ ]:
har: Path = nus333_dir()
p = GTAPParameters()
p.load_from_har(
    basedata_path=har / "basedata.har",
    sets_path=har / "sets.har",
    default_path=har / "climatechange.prm",
    baserate_path=har / "baserate.har",
)

## 2. El cierre de la línea de base, con `@overwrite`

- **`ShockBlock`**:
  - `axp[r,a]`, la productividad de cada actividad, deja de ser un instrumento fijo y
    pasa a ser endógena;
  - `b.target` declara el objetivo `gdp_target` sobre el PIB real `rgdpmp`: una
    variable nueva, fija en 1, y la fila `rgdpmp[shock] = gdp_target × rgdpmp[check]`.
- **`ClosureBlock`**: `eq_aoreg`, `axp[r,a] = axp[r,AGR]` en MFG y SER. Las 3
  actividades de una región se mueven juntas: un solo shifter regional, como `aoreg`.

Por región se liberan 3 incógnitas y se agregan 3 filas, así que el sistema sigue
cuadrado. Con `period="shock"` el cierre cambia **sólo en el shock**: en base y check la
productividad queda fija. Importa aquí: con `climatechange.prm` el check no reproduce
la base (el PIB real de ROW sale +0,03%), y el objetivo, como en GAMS, se mide contra
el check.

In [ ]:
REGS = ("USA", "ROW")
ACTS = ("AGR", "MFG", "SER")


@overwrite(ShockBlock, period="shock")
def baseline(b):
    for r in REGS:
        for a in ACTS:
            b.endogenous("axp", (r, a))
        b.target("gdp_target", (r,), quantity=lambda m, r: m.rgdpmp[r], domains=("r",))


@overwrite(ClosureBlock, period="shock")
def uniform_productivity(b):
    for r in REGS:
        for a in ("MFG", "SER"):
            b.equation(
                "eq_aoreg",
                (r, a),
                lambda m, r, a: m.axp[r, a] == m.axp[r, "AGR"],
                domains=("r", "a"),
            )

## 3. Cierre global y modelo

In [ ]:
gc = GTAPClosureConfig(
    name="base",
    closure_type="MCP",
    capital_mobility="sluggish",
    fix_endowments=False,
    fix_taxes=False,
    fix_technology=False,
    if_sub=False,
    savf_flag="capFlex",
    numeraire="pnum",
)
m, _ = build_block_model(p, p.sets, gc, "ROW", base_calibrated=False, ref_gdx=None)
for t in ("check", "shock"):
    print(f"axp[USA,AGR] libre en {t}:", not m.axp["USA", "AGR", t].fixed)
print("filas eq_gdp_target:", list(m.eq_gdp_target))

## 4. Los shocks

El PIB real como factor sobre el check; población y dotaciones de factores como en
`ME9A.EXP` (`qe` de GEMPACK es `aft` en equilibria).

In [ ]:
for r, pct in {"USA": 109.6, "ROW": 284.5}.items():
    fix_instrument_shock(m, "gdp_target", (r,), factor=1 + pct / 100)
for r, pct in {"USA": 32.3, "ROW": 37.3}.items():
    fix_instrument_shock(m, "pop", (r,), factor=1 + pct / 100)
QE = {
    ("USA", "LABOR"): 24.1, ("ROW", "LABOR"): 38.4,
    ("USA", "CAPITAL"): 60.6, ("ROW", "CAPITAL"): 213.1,
    ("USA", "LAND"): -0.93, ("ROW", "LAND"): 4.4,
}
for k, pct in QE.items():
    fix_instrument_shock(m, "aft", k, factor=1 + pct / 100)

## 5. Resolver check y shock

In [ ]:
res = solve_multiperiod(
    m,
    p,
    gc,
    ref_gdx=None,
    skip_base_solve=True,  # GAMS no resuelve la base
    mute_welfare=True,
    seed_from_prior=False,
    mode="gtap",
    solve_check=True,
)
print("codes:", {k: int(res[k]["code"]) for k in res})

## 6. Resultados (% de check a shock) contra GAMS y GEMPACK

In [ ]:
def pct(var, *k):
    comp = getattr(m, var)
    return 100.0 * (value(comp[(*k, "shock")]) / value(comp[(*k, "check")]) - 1.0)


filas = [
    ("PIB real USA (rgdpmp)", pct("rgdpmp", "USA"), 109.6, 109.6),
    ("PIB real ROW (rgdpmp)", pct("rgdpmp", "ROW"), 284.5, 284.5),
    ("productividad USA (aoreg)", pct("axp", "USA", "AGR"), 31.9496, 31.9376),
    ("productividad ROW (aoreg)", pct("axp", "ROW", "AGR"), 42.4131, 42.3080),
]
print(f"{'':28s}{'equilibria':>12s}{'GAMS':>12s}{'GEMPACK':>12s}")
for nombre, eq, gams, gem in filas:
    print(f"{nombre:28s}{eq:+12.4f}{gams:+12.4f}{gem:+12.4f}")

equilibria y GAMS coinciden en todas las celdas. GEMPACK pide 0,01pp menos de
productividad en USA y 0,11pp menos en ROW. No es el cierre: con la **misma**
productividad (ME9B fija `aoreg` en 31,94 / 42,31) GAMS y GEMPACK ya dan PIB distintos
(USA +109,58 contra +109,45; ROW +284,06 contra +284,32).